# Ticket Cancellation Prediction

**Selected Machine Learning Exercise**

Binary classification project for predicting whether a booked ticket will be cancelled.

In [1]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("data/train_data.csv")
df.head()

In [3]:
df.shape

(101017, 22)

In [ ]:
test = pd.read_csv("data/test_data.csv")
test.head()

In [5]:
test.shape

(43293, 20)

In [6]:
df.duplicated().sum()

np.int64(1)

In [7]:
df = df.drop_duplicates()

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
x = df.drop(columns=["Cancel", "CancelTime"])
y = df["Cancel"]

In [10]:
x.head()

,Created,DepartureTime,BillID,TicketID,ReserveStatus,UserID,Male,Price,CouponDiscount,From,To,Domestic,VehicleType,VehicleClass,TripReason,Vehicle,HashPassportNumber_p,HashEmail,BuyerMobile,NationalCode
0,2022-07-26 13:33:20.457,2022-07-26 16:30:00,38428546,7445571.0,3,NaN,True,1180000.0,0.0,شیراز,سیرجان,1,VIPمانیتوردار-شارژراختصاصی تخت شو مارال (جدید)...,True,Work,Bus,NaN,NaN,302222356019,330024570
1,2022-10-27 23:07:01.837,2022-10-29 09:45:00,39768762,7762719.0,5,NaN,False,1050000.0,0.0,قم,ساری,1,classicus 2+2,True,Int,Bus,NaN,NaN,900764168521,995520696
2,2022-09-12 11:01:13.607,2022-10-03 18:35:00,39128001,2327596.0,5,800398.0,False,4674000.0,0.0,تهران,ارومیه,1,فوکر 100,False,Int,Plane,NaN,1c44d7a76b52341fa12dcfa993138576befcc9ebf01d14...,749804783291,979382950
3,2022-08-08 17:43:35.840,2022-08-08 22:30:00,38606546,7495440.0,3,NaN,True,1200000.0,0.0,تهران,ملایر,1,VIPدرسا+مانیتوردار+شارژراختصاصی+پذیرایی,True,Work,Bus,NaN,NaN,781396205677,911237229
4,2022-11-01 15:12:56.823,2022-11-03 11:30:00,39822185,2356902.0,5,NaN,True,6222000.0,0.0,چابهار,تهران,1,NaN,False,Work,Plane,NaN,bb38b345aec02255e31d178492907175c5984f2a1f5b59...,524576220177,727496008


# feature engineering

In [11]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

## handling missing values

In [12]:
x_train.isna().sum()

Created                     0
DepartureTime               0
BillID                      0
TicketID                    0
ReserveStatus               0
UserID                  46827
Male                        0
Price                       0
CouponDiscount              0
From                        0
To                          0
Domestic                    0
VehicleType              6162
VehicleClass            30626
TripReason                  0
Vehicle                     0
HashPassportNumber_p    80111
HashEmail               46329
BuyerMobile                 0
NationalCode                0
dtype: int64

In [13]:
x_train.isna().mean().mul(100)

Created                  0.000000
DepartureTime            0.000000
BillID                   0.000000
TicketID                 0.000000
ReserveStatus            0.000000
UserID                  57.945602
Male                     0.000000
Price                    0.000000
CouponDiscount           0.000000
From                     0.000000
To                       0.000000
Domestic                 0.000000
VehicleType              7.625105
VehicleClass            37.897837
TripReason               0.000000
Vehicle                  0.000000
HashPassportNumber_p    99.132555
HashEmail               57.329357
BuyerMobile              0.000000
NationalCode             0.000000
dtype: float64

In [14]:
for df in [x_train, x_val, test]:
    df.drop(columns=["HashPassportNumber_p"], inplace=True)

In [15]:
x_train.isna().mean().mul(100)

Created            0.000000
DepartureTime      0.000000
BillID             0.000000
TicketID           0.000000
ReserveStatus      0.000000
UserID            57.945602
Male               0.000000
Price              0.000000
CouponDiscount     0.000000
From               0.000000
To                 0.000000
Domestic           0.000000
VehicleType        7.625105
VehicleClass      37.897837
TripReason         0.000000
Vehicle            0.000000
HashEmail         57.329357
BuyerMobile        0.000000
NationalCode       0.000000
dtype: float64

* `UserID`: creating features with it then dropping it

In [16]:
x_train["UserID"].value_counts(dropna=False)

UserID
NaN         46827
831022.0      104
228918.0       73
186339.0       69
853057.0       67
            ...  
217004.0        1
951894.0        1
826073.0        1
868338.0        1
590532.0        1
Name: count, Length: 13684, dtype: int64

In [17]:
id_cols = [
    "UserID",
    "BillID",
    "HashEmail",
    "BuyerMobile",
    "NationalCode"]

# فقط روی train یاد بگیر
freq_maps = {col: x_train[col].value_counts(dropna=True) for col in id_cols}

def add_id_features(df, freq_maps):
    df = df.copy()

    for col, freq in freq_maps.items():

        # چند بار این ID در train دیده شده؟
        df[f"{col}_count"] = df[col].map(freq).fillna(0).astype(int)

        # آیا تکراری است؟
        df[f"{col}_repeated"] = (df[f"{col}_count"] > 1).astype(int)

    return df


x_train = add_id_features(x_train, freq_maps)
x_val   = add_id_features(x_val, freq_maps)
test  = add_id_features(test, freq_maps)

In [18]:
drop_cols = [
    "TicketID",
    "UserID",
    "BillID",
    "HashEmail",
    "BuyerMobile",
    "NationalCode"]

x_train.drop(columns=drop_cols, inplace=True)
x_val.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

In [19]:
x_train.isna().mean().mul(100)

Created                   0.000000
DepartureTime             0.000000
ReserveStatus             0.000000
Male                      0.000000
Price                     0.000000
CouponDiscount            0.000000
From                      0.000000
To                        0.000000
Domestic                  0.000000
VehicleType               7.625105
VehicleClass             37.897837
TripReason                0.000000
Vehicle                   0.000000
UserID_count              0.000000
UserID_repeated           0.000000
BillID_count              0.000000
BillID_repeated           0.000000
HashEmail_count           0.000000
HashEmail_repeated        0.000000
BuyerMobile_count         0.000000
BuyerMobile_repeated      0.000000
NationalCode_count        0.000000
NationalCode_repeated     0.000000
dtype: float64

* `VehicleType`: filling with unknown

In [20]:
x_train["VehicleType"].value_counts(dropna=False)

VehicleType
NaN                                                          6162
4 ستاره اتوبوسي صبا                                          4907
3 ستاره 6 تخته پارسي                                         2403
25 نفره (VIP)                                                1774
4 ستاره 4 تخته غزال                                          1604
                                                             ... 
مان (ماهان ) VIP مانیتوردار.همراه باپک بهداشتی  رویال سفر       1
درسامانیتور دار اختصاصی VIP                                     1
درسا-مجهز به مانیتور و شارژر اختصاصی مدل201                     1
اسکانیا مارال همراه باپک بهداشتی                                1
VIP تک صندلی۲۹نفره همراه باپذیرایی اسیاسفر                      1
Name: count, Length: 2962, dtype: int64

In [21]:
for df in [x_train, x_val, test]:
    df["VehicleType"] = df["VehicleType"].fillna("Unknown")

* `VehicleClass`: correlated with VehicleType

In [22]:
x_train["VehicleType"].value_counts(dropna=False)

VehicleType
Unknown                                                      6162
4 ستاره اتوبوسي صبا                                          4907
3 ستاره 6 تخته پارسي                                         2403
25 نفره (VIP)                                                1774
4 ستاره 4 تخته غزال                                          1604
                                                             ... 
مان (ماهان ) VIP مانیتوردار.همراه باپک بهداشتی  رویال سفر       1
درسامانیتور دار اختصاصی VIP                                     1
درسا-مجهز به مانیتور و شارژر اختصاصی مدل201                     1
اسکانیا مارال همراه باپک بهداشتی                                1
VIP تک صندلی۲۹نفره همراه باپذیرایی اسیاسفر                      1
Name: count, Length: 2962, dtype: int64

In [23]:
pd.crosstab(
    x_train["VehicleType"],
    x_train["VehicleClass"],
    normalize="index"
).sort_values(by=True, ascending=False)

VehicleClass,False,True
VehicleType,,
۳۲VIPنفره,0.0,1.0
,0.0,1.0
( مانیتوردارNEW FACE)مان,0.0,1.0
((( وی ای پی اسکانیا))),0.0,1.0
(((اسکانیا وی ای پی))) تخت شو ((۲۵))نفره رشت,0.0,1.0
...,...,...
BENZ C457 2+2 / سیستم تهویه مطبوع,1.0,0.0
BENZ ***457 از مسیر فیروزکوه,1.0,0.0
CLASSICUS 2+2 / یک وعده غذای گرم,1.0,0.0


In [24]:
pd.crosstab(x_train["VehicleType"], x_train["VehicleClass"])

VehicleClass,False,True
VehicleType,,
,0,30
( مانیتوردارNEW FACE)مان,0,19
((( وی ای پی اسکانیا))),0,9
(((اسکانیا وی ای پی))) تخت شو ((۲۵))نفره رشت,0,4
(((اسکانیا وی ای پی))) تخت شو ((۲۵))نفره رشت میدان گیل ورودی پایانه,0,4
...,...,...
۳۰نفره تک صندلی مارالVIP (همسفر),0,3
۳۲VIPنفره,0,3
۴۴ نفره,1,0


In [25]:
vehicle_class_map = (
    x_train
    .dropna(subset=["VehicleClass", "VehicleType"])
    .groupby("VehicleType")["VehicleClass"]
    .agg(lambda x: x.mode().iloc[0])
)

# fallback فقط از train
vehicle_class_mode = x_train["VehicleClass"].mode().iloc[0]


for df in [x_train, x_val, test]:

    # اول با VehicleType
    df["VehicleClass"] = df["VehicleClass"].fillna(
        df["VehicleType"].map(vehicle_class_map)
    )

    # هرچی باقی موند
    df["VehicleClass"] = df["VehicleClass"].fillna(
        vehicle_class_mode
    )

In [26]:
x_train.isna().mean().mul(100)

Created                  0.0
DepartureTime            0.0
ReserveStatus            0.0
Male                     0.0
Price                    0.0
CouponDiscount           0.0
From                     0.0
To                       0.0
Domestic                 0.0
VehicleType              0.0
VehicleClass             0.0
TripReason               0.0
Vehicle                  0.0
UserID_count             0.0
UserID_repeated          0.0
BillID_count             0.0
BillID_repeated          0.0
HashEmail_count          0.0
HashEmail_repeated       0.0
BuyerMobile_count        0.0
BuyerMobile_repeated     0.0
NationalCode_count       0.0
NationalCode_repeated    0.0
dtype: float64

## creating features

In [27]:
for df in [x_train, x_val, test]:
    df["Created"] = pd.to_datetime(df["Created"])
    df["DepartureTime"] = pd.to_datetime(df["DepartureTime"])

    df["HoursUntilDeparture"] = (
        (df["DepartureTime"] - df["Created"])
        .dt.total_seconds() / 3600
    )

In [28]:
for df in [x_train, x_val, test]:
    df["BookedLast24h"] = (df["HoursUntilDeparture"] <= 24).astype(int)
    df["BookedLast72h"] = (df["HoursUntilDeparture"] <= 72).astype(int)
    df["BookedLongBefore"] = (df["HoursUntilDeparture"] >= 7 * 24).astype(int)

In [29]:
for df in [x_train, x_val, test]:
    df["DepartureHour"] = df["DepartureTime"].dt.hour
    df["DepartureDayOfWeek"] = df["DepartureTime"].dt.dayofweek
    df["IsWeekend"] = df["DepartureDayOfWeek"].isin([3, 4]).astype(int)

In [30]:
for df in [x_train, x_val, test]:
    df["CreatedHour"] = df["Created"].dt.hour
    df["CreatedDayOfWeek"] = df["Created"].dt.dayofweek

In [31]:
for df in [x_train, x_val, test]:
    df["DiscountRatio"] = (
        df["CouponDiscount"] / (df["Price"] + df["CouponDiscount"] + 1)
    )
    df["HasDiscount"] = (df["CouponDiscount"] > 0).astype(int)

In [32]:
route_counts = x_train.groupby(["From", "To"]).size()

for df in [x_train, x_val, test]:
    routes = pd.MultiIndex.from_frame(df[["From", "To"]])

    df["RouteFrequency"] = (
        route_counts
        .reindex(routes)
        .fillna(0)
        .to_numpy()
    )

In [33]:
route_median_price = (
    x_train
    .groupby(["From", "To"])["Price"]
    .median()
)

for df in [x_train, x_val, test]:
    routes = pd.MultiIndex.from_frame(df[["From", "To"]])

    median_price = route_median_price.reindex(routes).to_numpy()

    df["PriceVsRouteMedian"] = (
        df["Price"] / (median_price + 1)
    )

In [34]:
global_price_median = x_train["Price"].median()

for df in [x_train, x_val, test]:

    fallback = (
        df["Price"] / (global_price_median + 1)
    )

    df["PriceVsRouteMedian"] = (
        df["PriceVsRouteMedian"]
        .fillna(fallback)
    )

* date-time features

In [35]:
import holidays

In [36]:
iran_holidays = holidays.country_holidays(
    "IR",
    years=[2022, 2023]
)

holiday_dates = set(iran_holidays.keys())


def add_datetime_features(df):
    df = df.copy()

    df["Created"] = pd.to_datetime(df["Created"])
    df["DepartureTime"] = pd.to_datetime(df["DepartureTime"])

    # فاصله خرید تا حرکت
    df["HoursUntilDeparture"] = (
        (df["DepartureTime"] - df["Created"])
        .dt.total_seconds() / 3600
    )

    # زمان خرید
    df["CreatedHour"] = df["Created"].dt.hour
    df["CreatedDayOfWeek"] = df["Created"].dt.dayofweek
    df["CreatedMonth"] = df["Created"].dt.month

    # زمان حرکت
    df["DepartureHour"] = df["DepartureTime"].dt.hour
    df["DepartureDayOfWeek"] = df["DepartureTime"].dt.dayofweek
    df["DepartureMonth"] = df["DepartureTime"].dt.month

    # تعطیلات رسمی ایران
    departure_date = df["DepartureTime"].dt.date

    df["IsOfficialHoliday"] = (
        departure_date.isin(holiday_dates)
    ).astype(int)

    # پنجشنبه / جمعه
    df["IsThursday"] = (
        df["DepartureDayOfWeek"] == 3
    ).astype(int)

    df["IsFriday"] = (
        df["DepartureDayOfWeek"] == 4
    ).astype(int)

    # آخر هفته
    df["IsWeekend"] = (
        (df["IsThursday"] == 1) |
        (df["IsFriday"] == 1)
    ).astype(int)

    # تعطیلی برای سفر:
    # تعطیل رسمی یا جمعه
    df["IsHoliday"] = (
        (df["IsOfficialHoliday"] == 1) |
        (df["IsFriday"] == 1)
    ).astype(int)

    return df

In [37]:
x_train = add_datetime_features(x_train)
x_val = add_datetime_features(x_val)
test = add_datetime_features(test)

In [38]:
for df in [x_train, x_val, test]:
    df.drop(
        columns=["Created", "DepartureTime"],
        inplace=True
    )

In [39]:
north_cities = [
    "آستارا", "بندرانزلی", "رشت", "رودسر", "صومعه سرا",
    "طوالش", "فومن", "لاهیجان (گیلان )", "لنگرود", "ماسال",

    "آمل", "بابل", "بابلسر", "بهشهر", "تنکابن", "رامسر",
    "ساری", "شیرگاه", "عباس آباد(مازندران )", "قائمشهر",
    "قایم شهر", "محمودآباد (مازندران )", "نور", "نوشهر",
    "نکا", "پل سفید", "چالوس",

    "بندر ترکمن", "کلاله", "گرگان", "گنبدکاووس"
]

for df in [x_train, x_val, test]:
    df["IsNorthDestination"] = (
        df["To"].isin(north_cities)
    ).astype(int)

In [40]:
for df in [x_train, x_val, test]:
    df["NorthWeekendTrip"] = (
        (df["IsNorthDestination"] == 1) &
        (df["IsWeekend"] == 1)
    ).astype(int)

In [41]:
x_train.head()

,ReserveStatus,Male,Price,CouponDiscount,From,To,Domestic,VehicleType,VehicleClass,TripReason,...,RouteFrequency,PriceVsRouteMedian,CreatedMonth,DepartureMonth,IsOfficialHoliday,IsThursday,IsFriday,IsHoliday,IsNorthDestination,NorthWeekendTrip
76853,3,True,830000.0,0.0,بابلسر,تهران,1,درسا VIP مانیتوردار,True,Work,...,106,0.768518,4,4,0,0,0,0,0,0
10850,3,False,940000.0,0.0,تهران,زنجان,1,مارال۲۵نفره(تخت شو),True,Work,...,485,1.088592,4,4,0,0,0,0,0,0
2442,3,True,1300000.0,0.0,شیراز,عسلويه,1,VIP 2+1 / پذیرایی / سیستم تهویه مطبوع / تخت شو,True,Work,...,264,0.999999,9,9,0,0,1,1,0,0
52074,3,True,1220000.0,3630.0,تهران,اصفهان,1,مان VIP (کاوه),True,Work,...,1705,0.999999,10,11,0,0,0,0,0,0
34701,3,True,770000.0,0.0,تهران,آمل,1,مان 26 نفره VIP,True,Int,...,24,1.305083,10,10,0,0,1,1,1,1


In [42]:
for df in [x_train, x_val, test]:
    df.drop(
        columns=["From", "To"],
        inplace=True
    )

In [ ]:
x_train.columns.to_frame()

## creating pipelines

In [44]:
cyclic_periods = {
    "DepartureHour": 24,
    "DepartureDayOfWeek": 7,
    "DepartureMonth": 12,

    "CreatedHour": 24,
    "CreatedDayOfWeek": 7,
    "CreatedMonth": 12,
}

In [45]:
import numpy as np

for df in [x_train, x_val, test]:
    for col, period in cyclic_periods.items():

        df[f"{col}_sin"] = np.sin(
            2 * np.pi * df[col] / period
        )

        df[f"{col}_cos"] = np.cos(
            2 * np.pi * df[col] / period
        )

In [46]:
for df in [x_train, x_val, test]:
    df.drop(
        columns=list(cyclic_periods.keys()),
        inplace=True
    )

In [47]:
# numeric = [
#     "Price",
#     "CouponDiscount",

#     "UserID_count",
#     "BillID_count",
#     "HashEmail_count",
#     "BuyerMobile_count",
#     "NationalCode_count",

#     "HoursUntilDeparture",
#     "DiscountRatio",
#     "RouteFrequency",
#     "PriceVsRouteMedian",

#     # cyclic
#     "DepartureHour_sin",
#     "DepartureHour_cos",
#     "DepartureDayOfWeek_sin",
#     "DepartureDayOfWeek_cos",
#     "DepartureMonth_sin",
#     "DepartureMonth_cos",

#     "CreatedHour_sin",
#     "CreatedHour_cos",
#     "CreatedDayOfWeek_sin",
#     "CreatedDayOfWeek_cos",
#     "CreatedMonth_sin",
#     "CreatedMonth_cos",
# ]

In [48]:
# boolean = [
#     "Male",
#     "Domestic",
#     "VehicleClass",

#     "UserID_repeated",
#     "BillID_repeated",
#     "HashEmail_repeated",
#     "BuyerMobile_repeated",
#     "NationalCode_repeated",

#     "BookedLast24h",
#     "BookedLast72h",
#     "BookedLongBefore",

#     "HasDiscount",

#     "IsWeekend",
#     "IsOfficialHoliday",
#     "IsThursday",
#     "IsFriday",
#     "IsHoliday",

#     "IsNorthDestination",
#     "NorthWeekendTrip",
# ]

In [49]:
# cat_1hot = [
#     "TripReason",
#     "Vehicle",
# ]

In [50]:
# cat_high_card = [
#     "VehicleType"
# ]

In [51]:
cat_cols = [
    "TripReason",
    "Vehicle",
    "VehicleType",
    "ReserveStatus"
]

In [52]:
for df in [x_train, x_val, test]:
    for col in cat_cols:
        df[col] = df[col].astype(str)

In [53]:
# for df in [x_train, x_val, test]:
#     df.drop(columns=["ReserveStatus"], inplace=True)

### CatBoost dependency

CatBoost is listed in `requirements.txt`. Install the project dependencies before running the notebook.

In [55]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

from catboost import CatBoostClassifier


In [56]:
model = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100
)

In [57]:
model.fit(
    x_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(x_val, y_val),
    early_stopping_rounds=100
)

0:	test: 0.9926059	best: 0.9926059 (0)	total: 212ms	remaining: 3m 31s
100:	test: 0.9960774	best: 0.9960774 (100)	total: 7.25s	remaining: 1m 4s
200:	test: 0.9964185	best: 0.9964185 (200)	total: 14.8s	remaining: 59s
300:	test: 0.9965995	best: 0.9965995 (300)	total: 22.2s	remaining: 51.5s
400:	test: 0.9966766	best: 0.9966766 (400)	total: 29.5s	remaining: 44s
500:	test: 0.9967426	best: 0.9967439 (499)	total: 36.8s	remaining: 36.7s
600:	test: 0.9967242	best: 0.9967619 (536)	total: 44.1s	remaining: 29.2s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9967619343
bestIteration = 536

Shrink model to first 537 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=8, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [58]:
y_prob = model.predict_proba(x_val)[:, 1]

best_threshold = 0
best_f1 = 0

for threshold in np.arange(0.05, 0.96, 0.01):
    y_pred = (y_prob >= threshold).astype(int)
    score = f1_score(y_val, y_pred)

    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)

Best threshold: 0.6400000000000001
Best F1: 0.9536692593845113


In [59]:
best_iteration = model.get_best_iteration()

if best_iteration <= 0:
    best_iteration = 1000
else:
    best_iteration += 1

print("Best iteration:", best_iteration)
print("Best threshold:", best_threshold)
print("Validation F1:", best_f1)

Best iteration: 537
Best threshold: 0.6400000000000001
Validation F1: 0.9536692593845113


In [60]:
import pandas as pd
from catboost import CatBoostClassifier

x_full = pd.concat(
    [x_train, x_val],
    ignore_index=True
)

y_full = pd.concat(
    [y_train, y_val],
    ignore_index=True
)

In [61]:
final_model = CatBoostClassifier(
    iterations=best_iteration,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100
)

final_model.fit(
    x_full,
    y_full,
    cat_features=cat_cols
)

0:	total: 101ms	remaining: 54s
100:	total: 12.4s	remaining: 53.6s
200:	total: 21.1s	remaining: 35.4s
300:	total: 32.5s	remaining: 25.4s
400:	total: 40.9s	remaining: 13.9s
500:	total: 53.7s	remaining: 3.86s
536:	total: 57.8s	remaining: 0us


CatBoostClassifier(auto_class_weights='Balanced', depth=8, eval_metric='AUC', iterations=537, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [62]:
test_prob = final_model.predict_proba(test)[:, 1]

test_pred = (
    test_prob >= best_threshold
).astype(int)

In [ ]:
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)

submission = pd.DataFrame({
    "Cancel": test_pred
})

submission.to_csv("outputs/submission.csv", index=False)

In [ ]:
final_model.save_model("outputs/model.cbm")

In [ ]:
import zipfile

artifact_files = ["outputs/submission.csv", "outputs/model.cbm"]

with zipfile.ZipFile("outputs/model_artifacts.zip", mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_name in artifact_files:
        zf.write(file_name, arcname=Path(file_name).name)

print("Created outputs/model_artifacts.zip")